# Table of Contents

1. Text Cleaning and Normalization Pipelines
2. Bag-of-Words and TF-IDF Vectorization
3. Character and Word Length Metrics
4. Basic Sentiment Scoring Text Extraction

In [8]:
# Global Imports and Runtime Visual Configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler,MinMaxScaler,RobustScaler,PowerTransformer,QuantileTransformer
from sklearn.feature_extraction.text import CountVectorizer
# Visual styling settings
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 10

print("Global libraries for categorical encoding strategies loaded successfully.")

Global libraries for categorical encoding strategies loaded successfully.


---

## 1. Text Cleaning and Normalization Pipelines

### 1. Definition

Text cleaning and normalization pipelines standardize raw text strings by converting cases, stripping punctuation, removing stop words, and applying tokenization or lemmatization to reduce feature sparsity.

### 2. Regular Expression

`[^\w\s]`

- `[]` : Defines a character set.
- `^` : Negation (matches characters NOT included in the set).
- `\w` : Matches word characters (letters, digits, and underscore).
- `\s` : Matches whitespace characters (spaces, tabs, newlines).
- `[^\w\s]` : Matches all characters except word characters and whitespace, such as `#`, `!`, `@`, `$`, `%`, etc.

### 3. Naive Example

```python
import re

text = "Hello!!! Welcome @Python #MachineLearning"

cleaned_text = re.sub(r'[^\w\s]', '', text)

print(cleaned_text)

In [9]:
df_text = pd.DataFrame({
    'raw_text':['Great Product!!!', 'Terrible Services....', 'ABSOLUTE garbage!']
})
print(df_text)
df_text['cleaned_text'] = df_text['raw_text'].str.lower().str.replace(r'[^\w\s]','',regex=True)
print(df_text)

                raw_text
0       Great Product!!!
1  Terrible Services....
2      ABSOLUTE garbage!
                raw_text       cleaned_text
0       Great Product!!!      great product
1  Terrible Services....  terrible services
2      ABSOLUTE garbage!   absolute garbage


---

## 2. Bag-of-Words and TF-IDF Vectorization

### 1. Definition

Bag-of-Words (BoW) counts term frequencies in documents, whereas Term Frequency-Inverse Document Frequency (TF-IDF) weights terms by their local frequency and penalizes globally ubiquitous words across the corpus.

### 2. Mathematical Formula

**Term Frequency (TF):**

$$
TF(t,d) = \frac{\text{Count of term }t\text{ in document }d}{\text{Total terms in document }d}
$$

**Inverse Document Frequency (IDF):**

$$
IDF(t) = \log\left(\frac{N}{DF(t)}\right)
$$

**TF-IDF:**

$$
TFIDF(t,d) = TF(t,d) \times IDF(t)
$$

Where:
- \(t\) = Term (word).
- \(d\) = Document.
- \(N\) = Total number of documents.
- \(DF(t)\) = Number of documents containing term \(t\).

### 3. Naive Example

Vectorizing customer reviews so that common words like 'the' receive low weights while informative words like 'faulty' receive high importance scores.

### 4. Uses

- Converting unstructured text documents into numerical feature matrices for classification.
- Information retrieval and document similarity search.
- Keyword extraction.

In [10]:
corpus = [
    'Macine Learning is Great',
    'Machine Learning is fun and learning is powerful'
]

vectorizer = CountVectorizer()
x = vectorizer.fit_transform(corpus)

print("Vocabulary: ",vectorizer.get_feature_names_out())
print("TF-IDF Scores Matrix:\n",x.toarray())

Vocabulary:  ['and' 'fun' 'great' 'is' 'learning' 'machine' 'macine' 'powerful']
TF-IDF Scores Matrix:
 [[0 0 1 1 1 0 1 0]
 [1 1 0 2 2 1 0 1]]


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

# 1. Sample text corpus
corpus = [
    "Machine learning is great",
    "Machine learning is fun and learning is powerful"
]

# 2. Initialize and fit TfidfVectorizer
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

# 3. View results
print("Vocabulary:", vectorizer.get_feature_names_out())
print("TF-IDF Scores Matrix:\n", X.toarray())

Vocabulary: ['and' 'fun' 'great' 'is' 'learning' 'machine' 'powerful']
TF-IDF Scores Matrix:
 [[0.         0.         0.63009934 0.44832087 0.44832087 0.44832087
  0.        ]
 [0.36378803 0.36378803 0.         0.51767635 0.51767635 0.25883818
  0.36378803]]


In [12]:
corpus = ['fast delivery fast service', 'slow shipping and terrible service', 'excellent product quality']
tfidf = TfidfVectorizer()
tfidf_matrix = tfidf.fit_transform(corpus)
df_tfidf = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())
print(
    "TF-IDF Feature Matrix Summary:\n",
    df_tfidf.iloc[:, :4],
)

TF-IDF Feature Matrix Summary:
         and  delivery  excellent      fast
0  0.000000  0.423394    0.00000  0.846789
1  0.467351  0.000000    0.00000  0.000000
2  0.000000  0.000000    0.57735  0.000000


## 3 Character and Word Length Matrix

1. Definition

Character and word length metrics extract structural metadata features from raw text (such as character count, word count, and average word length) to capture writing style and verbosity.

In [13]:
df_len = pd.DataFrame({'text': ['Short review', 'This is a much longer and more detailed customer feedback review']})
df_len['char_count'] = df_len['text'].str.len()
df_len['word_count'] = df_len['text'].str.split().str.len()
print("Length Metrics DataFrame:\n", df_len)

Length Metrics DataFrame:
                                                 text  char_count  word_count
0                                       Short review          12           2
1  This is a much longer and more detailed custom...          64          11


## 4 Basics Sentiment Scoring Text Extraction

1. Definition
   
   Basic sentiment scoring extracts lexicon-based polarity scores by matching text tokens against predefined dictionaries of positive and negative sentiment weights.

In [14]:
positive_lexicon = {'great': 1, 'excellent': 2, 'good': 1}
negative_lexicon = {'terrible': -2, 'bad': -1, 'garbage': -2}


def compute_sentiment(text):
  score = 0
  for word in text.lower().split():
    score += positive_lexicon.get(word, 0) + negative_lexicon.get(word, 0)
  return score


df_sent = pd.DataFrame(
    {'review': ['great and excellent', 'terrible garbage', 'good product']}
)
df_sent['sentiment_score'] = df_sent['review'].apply(compute_sentiment)
print('Sentiment Scoring Results:\n', df_sent)

Sentiment Scoring Results:
                 review  sentiment_score
0  great and excellent                3
1     terrible garbage               -4
2         good product                1
